# 04 · Análisis Exploratorio (EDA)

> **Objetivo:** entender la *forma* de las features antes de modelar.
> El EDA cambia las decisiones de preprocesamiento.

## ¿Qué vamos a hacer?

1. Visualizar distribuciones de cada feature.
2. Identificar sesgo y outliers.
3. Mostrar el efecto de la transformación `log1p`.
4. Visualizar correlaciones.
5. Hacer un primer scatter plot (Recency vs Monetary) con tamaño = Frequency.

## ¿Por qué importa el EDA antes del clustering?

Porque las decisiones de preprocesamiento dependen de cómo se ven los datos:

- ¿Distribución sesgada? → log + escalar.
- ¿Outliers extremos? → robust scaler o transformaciones.
- ¿Variables en escalas muy distintas? → escalar (StandardScaler).
- ¿Variables muy correlacionadas? → considerar PCA.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import FEATURES_DATA_FILE
from src.visualization.plots import plot_distributions

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100


In [ ]:
features = pd.read_parquet(FEATURES_DATA_FILE)
print(f"Clientes: {len(features):,}")
features.describe().T


## 1. Distribuciones (histograma + boxplot)

La función `plot_distributions` del módulo de visualización dibuja
ambas gráficas lado a lado para cada variable.


In [ ]:
fig = plot_distributions(features, columns=["Recency", "Frequency", "Monetary"])
plt.show()


**Lo que deberías ver:**

- **Recency**: bimodal o uniforme. Hay clientes recientes y otros muy antiguos.
- **Frequency**: extremadamente sesgada a la derecha. La mayoría compró pocas
  veces; unos pocos compraron decenas o cientos.
- **Monetary**: peor que Frequency, long-tail brutal.

> **Errores comunes:**
>
> - Aplicar K-Means sobre `Monetary` sin transformar produce clusters
>   degenerados: un cluster diminuto con los mayoristas y otro gigante
>   con todo lo demás.
> - "Eliminar outliers" usando IQR rebana segmentos legítimos.

## 2. Efecto de log1p

`log1p(x) = log(1 + x)` comprime la cola larga sin hacer estallar el log
en los ceros. Es la transformación canónica para variables long-tail.


In [ ]:
log_features = features[["Recency", "Frequency", "Monetary"]].apply(np.log1p)
log_features.columns = [f"log_{c}" for c in log_features.columns]

fig = plot_distributions(log_features, columns=log_features.columns)
plt.show()


**Compara** estos histogramas con los anteriores. Las distribuciones
ahora se parecen más a campanas. K-Means y DBSCAN funcionan mucho mejor
con este input.

## 3. Correlaciones (heatmap)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    features.corr(),
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    square=True,
    ax=ax,
)
ax.set_title("Correlaciones entre features")
plt.tight_layout()
plt.show()


> **Observa:** `Frequency` ↔ `Monetary` ↔ `ProductDiversity` están
> muy correlacionadas. No es un problema para clustering, pero sí
> indica que podrías reducir dimensionalidad con PCA.

## 4. Scatter Recency vs Monetary, escala log

Una primera visualización del espacio de clientes:


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(
    features["Recency"],
    features["Monetary"],
    s=features["Frequency"] * 2,
    alpha=0.4,
    c=features["Frequency"],
    cmap="viridis",
)
ax.set_xlabel("Recency (días desde última compra)")
ax.set_ylabel("Monetary (GBP, log)")
ax.set_yscale("log")
ax.set_title("Espacio de clientes: Recency vs Monetary (tamaño y color = Frequency)")
plt.colorbar(sc, label="Frequency")
plt.tight_layout()
plt.show()


**Lectura:**

- Los clientes con **Recency baja + Monetary alta + Frequency alta** son
  los Champions (esquina inferior derecha del scatter).
- Los clientes con **Recency alta + Monetary baja** son inactivos.
- Hay puntos aislados arriba a la derecha → outliers / mayoristas.

## 5. Pairplot rápido (cuidado con datasets grandes)


In [ ]:
sample = features.sample(min(2000, len(features)), random_state=42)
sns.pairplot(np.log1p(sample[["Recency", "Frequency", "Monetary", "AvgTicket"]]),
             plot_kws={"alpha": 0.3, "s": 8}, height=2.2)
plt.suptitle("Pairplot (log) — muestra de 2000 clientes", y=1.02)
plt.show()


## Resumen

| Hallazgo | Implicación |
|---|---|
| Long-tail en Monetary y Frequency | Transformar con log1p antes de escalar. |
| Correlación alta RFM | OK para clustering; PCA opcional. |
| Outliers visibles | Conservarlos (mayoristas potenciales). |
| Distribución bimodal en Recency | Probable separación natural en clusters. |

---

## Preguntas de Reflexión

1. ¿Por qué `log1p(x)` es preferible a `log(x)` cuando hay valores cercanos a 0?
2. Si una feature tiene varianza casi cero, ¿qué efecto tiene en K-Means?
3. ¿Qué información se pierde al aplicar `log`? ¿Es importante esa pérdida?
4. ¿Detectaste algún subgrupo a simple vista en el scatter? ¿Cuántos clusters
   "esperarías" que K-Means encuentre?

> **Próximo paso:** ``05_preprocessing_pipelines.ipynb`` — formalizar
> las transformaciones en pipelines reproducibles.
